# Exploration des données CNPS

Statistiques descriptives sur la base salariale nettoyée (`cnps_cleaned.parquet`), puis jointure avec le référentiel entreprises ANSTAT pour comparer la nomenclature sectorielle CNPS à la nomenclature CEPICI.

**Sources des données :**
- `cnps_cleaned.parquet` : lu directement depuis MinIO (bucket `gold`, sortie de l'étape `03_nettoyage_donnees.py` du pipeline — voir `config/settings.yaml`, section `minio:`, clés `cleaned_bucket` / `cleaned_prefix`). Aucun fichier local requis pour cette partie.
- `REQUETES ANSTAT_MODULE EMPLOYEURS.xlsx` : fichier local, référentiel externe (nomenclature CEPICI) hors du pipeline CNPS, non géré par MinIO.

In [ ]:
import sys
from pathlib import Path

import polars as pl

# Rend le package "cnps" importable depuis le notebook (situe a la racine du repo)
sys.path.insert(0, str(Path.cwd() / "src"))

from cnps.config import load_config
from cnps.storage import read_parquet, write_parquet

ANSTAT_PATH = "REQUETES ANSTAT_MODULE EMPLOYEURS.xlsx"

# Charge config/settings.yaml + config/dimensions.yaml + les identifiants MinIO (.env)
cfg = load_config()
minio_cfg = cfg.minio

CLEANED_BUCKET = minio_cfg.cleaned_bucket
CLEANED_PREFIX = minio_cfg.cleaned_prefix
CLEANED_OBJECT = f"{CLEANED_PREFIX}cnps_cleaned.parquet"

print(f"MinIO endpoint : {minio_cfg.endpoint}")
print(f"Base nettoyee  : {CLEANED_BUCKET}/{CLEANED_OBJECT}")

pl.Config.set_tbl_rows(30)

## 1. Chargement de la base nettoyée depuis MinIO

In [ ]:
df = read_parquet(minio_cfg, CLEANED_BUCKET, CLEANED_OBJECT)

print(f"Lignes: {df.height:,}")
print(f"Colonnes: {df.width}")
df.head(10)

In [ ]:
# Valeurs manquantes et types par colonne
df.null_count().transpose(include_header=True, header_name="colonne", column_names=["n_null"]).with_columns(
    (pl.col("n_null") / df.height * 100).round(2).alias("pct_null")
).sort("pct_null", descending=True)

## 2. Statistiques descriptives ciblées

Plutôt qu'un profiling automatique exhaustif (coûteux sur ~27M lignes et peu lisible pour un usage répétable), on regarde ici les distributions des variables clés pour l'analyse salariale : `SALAIRE_BRUT_MENS` (variable d'intérêt), `AGE_EMPLOYE`, `EFFECTIF_SALARIES`, et la répartition des variables catégorielles principales.

In [ ]:
# Statistiques numériques clés
df.select("SALAIRE_BRUT_MENS", "AGE_EMPLOYE", "EFFECTIF_SALARIES", "ANCIENNETE_ENTREPRISE").describe()

In [ ]:
# Distribution du salaire mensuel par percentiles (pour juger de l'asymétrie / outliers résiduels)
df.select(
    pl.col("SALAIRE_BRUT_MENS").quantile(q).alias(f"p{int(q*100)}")
    for q in [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
)

In [ ]:
# Répartition des variables catégorielles principales
for col in ["SEXE", "SITUATION_MATRIMONIALE", "NIVEAU_ETUDE", "TYPE_SALARIE", "STATUT_TRAVAILLEUR", "CATEGORIE_ENTREPRISE"]:
    print(f"--- {col} ---")
    print(df[col].value_counts().sort("count", descending=True))
    print()

In [ ]:
# Évolution du nombre de déclarations et du salaire moyen par période (MOIS/ANNEE)
df.group_by("PERIOD").agg(
    pl.len().alias("n_declarations"),
    pl.col("ID_INDIV").n_unique().alias("n_individus"),
    pl.col("SALAIRE_BRUT_MENS").mean().round(0).alias("salaire_moyen"),
).sort("PERIOD")

## 3. Focus sur `SECTEUR_ACTIVITE` (nomenclature CNPS)

In [ ]:
# Répartition des effectifs et du salaire moyen par secteur (nomenclature CNPS)
secteur_stats = (
    df.group_by("SECTEUR_ACTIVITE")
    .agg(
        pl.len().alias("n_lignes"),
        pl.col("ID_EMPLOYEUR").n_unique().alias("n_entreprises"),
        pl.col("ID_INDIV").n_unique().alias("n_individus"),
        pl.col("SALAIRE_BRUT_MENS").mean().round(0).alias("salaire_moyen"),
        pl.col("SALAIRE_BRUT_MENS").median().round(0).alias("salaire_median"),
    )
    .sort("n_lignes", descending=True)
)
secteur_stats

In [ ]:
print("Pct SECTEUR_ACTIVITE manquant (CNPS):", round(df["SECTEUR_ACTIVITE"].null_count() / df.height * 100, 2), "%")
print("N secteurs distincts (CNPS):", df["SECTEUR_ACTIVITE"].n_unique())

## 4. Chargement du référentiel ANSTAT (`REQUETES ANSTAT_MODULE EMPLOYEURS.xlsx`)

Référentiel entreprises (feuille `DATA`) : raison sociale, secteur d'activité (nomenclature CEPICI, différente de la nomenclature CNPS), forme juridique, date de début d'activité, motif de radiation, etc. La feuille `SQL` ne contient que la requête source, pas de données à charger.

In [ ]:
df_anstat = pl.read_excel(ANSTAT_PATH, sheet_name="DATA")

print(f"Lignes: {df_anstat.height:,}")
print(f"Colonnes: {df_anstat.width}")
df_anstat.head(10)

In [ ]:
# Valeurs manquantes et secteurs distincts côté ANSTAT
print("N secteurs distincts (ANSTAT):", df_anstat["SECTEUR_ACTIVITE"].n_unique())
print("Pct SECTEUR_ACTIVITE manquant:", round(df_anstat["SECTEUR_ACTIVITE"].null_count() / df_anstat.height * 100, 2), "%")
print()
df_anstat["SECTEUR_ACTIVITE"].value_counts().sort("count", descending=True).head(20)

## 5. Jointure CNPS ↔ ANSTAT sur `RAISON_SOCIALE`

Il n'y a pas d'identifiant entreprise commun entre les deux fichiers (le `NUMERO_DFE`/`NUMERO_RCCM` de l'ANSTAT n'a pas d'équivalent direct confirmé côté CNPS) : la seule clé candidate est le nom d'entreprise. On normalise la raison sociale des deux côtés (majuscules, espaces, ponctuation) avant la jointure.

In [ ]:
def normalize_raison_sociale(col: pl.Expr) -> pl.Expr:
    return (
        col.str.to_uppercase()
        .str.replace_all(r"[^A-Z0-9 ]", "")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
    )

cnps_firms = (
    df.select("ID_EMPLOYEUR", "RAISON_SOCIALE")
    .unique()
    .with_columns(normalize_raison_sociale(pl.col("RAISON_SOCIALE")).alias("RAISON_SOCIALE_NORM"))
)

anstat_firms = df_anstat.with_columns(
    normalize_raison_sociale(pl.col("RAISON_SOCIALE")).alias("RAISON_SOCIALE_NORM")
)

print("Entreprises CNPS uniques:", cnps_firms.height)
print("Entreprises ANSTAT uniques (lignes):", anstat_firms.height)

In [ ]:
# Taux de correspondance (match exact sur raison sociale normalisée)
# suffixe "_anstat" sur les colonnes ANSTAT en collision de nom (RAISON_SOCIALE, SECTEUR_ACTIVITE)
matched = cnps_firms.join(anstat_firms, on="RAISON_SOCIALE_NORM", how="inner", suffix="_anstat")

n_matched = matched["ID_EMPLOYEUR"].n_unique()
n_total = cnps_firms["ID_EMPLOYEUR"].n_unique()

print(f"Entreprises CNPS matchées: {n_matched:,} / {n_total:,} ({n_matched / n_total * 100:.1f}%)")
matched.select("RAISON_SOCIALE", "RAISON_SOCIALE_NORM", "SECTEUR_ACTIVITE_anstat", "FORME JURIDIQUE").head(20)

In [ ]:
# Aperçu des entreprises CNPS non matchées (pour juger s'il faut un matching approché)
unmatched = cnps_firms.join(anstat_firms, on="RAISON_SOCIALE_NORM", how="anti")
print(f"Non matchées: {unmatched.height:,} ({unmatched.height / n_total * 100:.1f}%)")
unmatched.select("RAISON_SOCIALE").sample(n=20, seed=42)

## 6. Analyse des salaires par secteur : CNPS enrichi par le secteur ANSTAT

On construit une table de correspondance `ID_EMPLOYEUR -> SECTEUR_ACTIVITE_ANSTAT` (nomenclature CEPICI) à partir des entreprises matchées, puis on l'ajoute (`join`, pas de duplication de lignes) à la base salariale complète. Cela permet de comparer les statistiques de salaire selon la nomenclature CNPS *et* selon la nomenclature ANSTAT/CEPICI pour les entreprises où les deux sont disponibles.

In [ ]:
# Table de correspondance entreprise -> secteur ANSTAT (1 ligne par ID_EMPLOYEUR matché)
secteur_anstat_map = (
    matched.select("ID_EMPLOYEUR", pl.col("SECTEUR_ACTIVITE_anstat").alias("SECTEUR_ANSTAT"))
    .unique(subset="ID_EMPLOYEUR", keep="first")
)

# Jointure gauche : chaque ligne salariale CNPS garde son secteur CNPS, et gagne le secteur ANSTAT si connu
df_enrichi = df.join(secteur_anstat_map, on="ID_EMPLOYEUR", how="left")

print(f"Lignes CNPS enrichies: {df_enrichi.height:,} (identique à la base d'origine, {df.height:,})")
print("Pct lignes avec secteur ANSTAT connu:", round(df_enrichi["SECTEUR_ANSTAT"].is_not_null().sum() / df_enrichi.height * 100, 1), "%")

In [ ]:
# Salaire moyen/médian par secteur ANSTAT (nomenclature CEPICI), uniquement sur les lignes matchées
salaires_par_secteur_anstat = (
    df_enrichi.filter(pl.col("SECTEUR_ANSTAT").is_not_null())
    .group_by("SECTEUR_ANSTAT")
    .agg(
        pl.len().alias("n_lignes"),
        pl.col("ID_EMPLOYEUR").n_unique().alias("n_entreprises"),
        pl.col("ID_INDIV").n_unique().alias("n_individus"),
        pl.col("SALAIRE_BRUT_MENS").mean().round(0).alias("salaire_moyen"),
        pl.col("SALAIRE_BRUT_MENS").median().round(0).alias("salaire_median"),
        pl.col("SALAIRE_BRUT_MENS").std().round(0).alias("salaire_std"),
    )
    .sort("salaire_moyen", descending=True)
)
salaires_par_secteur_anstat

In [ ]:
# Comparaison directe : nomenclature CNPS vs nomenclature ANSTAT, pour les entreprises où les deux sont connus
comparaison_nomenclatures = (
    df_enrichi.filter(pl.col("SECTEUR_ANSTAT").is_not_null() & pl.col("SECTEUR_ACTIVITE").is_not_null())
    .group_by("SECTEUR_ACTIVITE", "SECTEUR_ANSTAT")
    .agg(
        pl.len().alias("n_lignes"),
        pl.col("SALAIRE_BRUT_MENS").mean().round(0).alias("salaire_moyen"),
    )
    .sort("n_lignes", descending=True)
)
comparaison_nomenclatures.head(30)

In [ ]:
# Évolution temporelle du salaire moyen pour les 5 plus gros secteurs ANSTAT (par nombre de lignes)
top5_secteurs = salaires_par_secteur_anstat.sort("n_lignes", descending=True).head(5)["SECTEUR_ANSTAT"].to_list()

evolution_secteurs = (
    df_enrichi.filter(pl.col("SECTEUR_ANSTAT").is_in(top5_secteurs))
    .group_by("PERIOD", "SECTEUR_ANSTAT")
    .agg(pl.col("SALAIRE_BRUT_MENS").mean().round(0).alias("salaire_moyen"))
    .sort("PERIOD", "SECTEUR_ANSTAT")
)
evolution_secteurs.pivot(index="PERIOD", on="SECTEUR_ANSTAT", values="salaire_moyen")

## 7. Export de la table enrichie (optionnel) vers MinIO

Décommenter pour sauvegarder la base salariale enrichie du secteur ANSTAT — utile pour reprendre l'analyse dans un autre outil (Excel, BI) sans recalculer la jointure. L'export se fait directement sur MinIO, aucun fichier n'est écrit sur le disque local.

In [ ]:
# write_parquet(minio_cfg, CLEANED_BUCKET, f"{CLEANED_PREFIX}cnps_enrichi_secteur_anstat.parquet", df_enrichi)
# write_parquet(minio_cfg, minio_cfg.output_bucket, f"{minio_cfg.output_prefix}salaires_par_secteur_anstat.parquet", salaires_par_secteur_anstat)